# Reusing code across notebooks

almond can't import a notebook from another notebook. What it can load are
[Ammonite scripts](https://ammonite.io/#ScalaScripts): `.sc` files with Scala code, like the ones in the
[`shared`](shared) directory next to this notebook:
- [`shared/geometry.sc`](shared/geometry.sc) defines a `Point` case class, its companion object, and a `distance` function,
- [`shared/shapes.sc`](shared/shapes.sc) loads `geometry.sc` itself, and defines a `Polygon` case class and a `square` function,
- [`shared/temperatures.sc`](shared/temperatures.sc) is the code of the [`shared/temperatures.ipynb`](shared/temperatures.ipynb)
  notebook, converted to a script (see [below](#Reusing-code-from-a-notebook)).

To share definitions between notebooks, put them in such scripts, and load the scripts from the notebooks that need them.
This notebook shows how, how to reuse the code of a notebook nonetheless, and what doesn't work.

## Loading scripts

Scripts can be loaded with a `//> using script` directive, or with an `import $file` statement:

In [1]:
//> using option "-Wconf:msg=permanently hidden:s"
//> using scripts shared/geometry.sc, shared/temperatures.sc
import $file.shared.shapes

geometry.sc loaded


import $file.$

- `//> using script` takes the path of a script, and `//> using scripts` takes several of them. Relative paths are relative
  to the working directory of the kernel, which is the directory of the notebook. Directives must be at the beginning of the cell.
- `//> using option "-Wconf:msg=permanently hidden:s"` silences spurious warnings
  (``imported `geometry` is permanently hidden by definition of value geometry``), that almond 0.15.0 prints in every cell run
  after one importing members of a script, like `import geometry._` [below](#Using-the-definitions).
- `import $file` takes the path as a Scala import: `/` becomes `.`, the `.sc` extension is dropped, and `^` stands for the
  parent directory (`import $file.^.common` loads `../common.sc`). In a script, like
  [`shared/shapes.sc`](shared/shapes.sc), these paths are relative to the script.

Each script is compiled and run once per session: `geometry.sc` printed `geometry.sc loaded` only once above,
even though `shared/shapes.sc` loads it too. If a script changes, loading it again re-compiles and re-runs it.

A script is identified by the path it's loaded with though, not by its file. If `geometry.sc` was next to this notebook,
loaded from here with `import $file.geometry`, and from `shared/shapes.sc` with `import $file.^.geometry`, it would be loaded
twice, as `geometry` and as `shared.^.geometry`, and the two `Point` classes would be different, incompatible, types.
Keeping the scripts that load each other in the same directory, like here, avoids that.

**Load all the scripts in the first code cell of the notebook.** With almond 0.15.0, a script loaded after other
cells were run works the first time, but fails with an error like
```text
object shapes is not a member of package ammonite.$sess
```
when the kernel is restarted and the notebook is run again, as the compiled script is then taken from a cache.

## Using the definitions

The definitions of a script are members of an object named after the script:

In [2]:
geometry.distance(geometry.Point.origin, geometry.Point(3, 4))

res2: Double = 5.0

Import them to use them without the prefix:

In [3]:
import geometry._
import shapes._

val sq = square(Point(1, 1), 2)
sq.perimeter

import geometry._
import shapes._
sq: Polygon = Polygon(
  vertices = ArraySeq(
    Point(x = 1.0, y = 1.0),
    Point(x = 3.0, y = 1.0),
    Point(x = 3.0, y = 3.0),
    Point(x = 1.0, y = 3.0)
  )
)
res3_3: Double = 8.0

In [4]:
sq.vertices.map(distance(Point.origin, _))

res4: Seq[Double] = ArraySeq(
  1.4142135623730951,
  3.1622776601683795,
  4.242640687119285,
  3.1622776601683795
)

## Reusing code from a notebook

There's no way to load the code of a notebook directly. But `jupyter nbconvert --to script` writes the code cells of a
notebook to a script, which can then be loaded like the other ones.

That's how [`shared/temperatures.sc`](shared/temperatures.sc), loaded in the first cell above, was written from
[`shared/temperatures.ipynb`](shared/temperatures.ipynb). The conversion can be run from a terminal, with
`jupyter nbconvert --to script shared/temperatures.ipynb`, or from Scala:

In [5]:
os.proc("jupyter", "nbconvert", "--to", "script", "shared/temperatures.ipynb").call(stderr = os.Pipe).exitCode

res5: Int = 0

In [6]:
print(os.read(os.pwd / "shared" / "temperatures.sc"))

case class Temperature(celsius: Double) {
  def fahrenheit: Double = celsius * 9 / 5 + 32
}

object Temperature {
  val freezing: Temperature = Temperature(0)
  val boiling: Temperature  = Temperature(100)
}

def describe(t: Temperature): String =
  f"${t.celsius}%.1f °C is ${t.fahrenheit}%.1f °F"

describe(Temperature.boiling)


Markdown cells are dropped, and the code cells are written one after the other. All of them run when the script is
loaded, so the notebooks you convert should mostly contain definitions, and nothing slow.

The definitions of the notebook can then be used like those of the other scripts:

In [7]:
import temperatures._

describe(Temperature(37))

import temperatures._
res7_1: String = "37.0 °C is 98.6 °F"

After changing `shared/temperatures.ipynb`, convert it again, then restart the kernel and run this notebook again.

## What doesn't work

`.ipynb` files can't be loaded as scripts. `//> using script` and `import $file` only load `.sc` files:

In [8]:
//> using script shared/temperatures.ipynb

Script not found: /home/jovyan/shared/temperatures.ipynb.sc

: Script not found: /home/jovyan/shared/temperatures.ipynb.sc

Some other things don't work with almond 0.15.0:
- loading a script with `interp.load.module(os.pwd / "shared" / "temperatures.sc")`, after other cells were run, has the same
  problem as `//> using script` and `import $file` [above](#Loading-scripts): it fails when the notebook is run again,
- Scala CLI's `//> using file` directive isn't supported, and fails with `Unrecognized directive: file with values: …`.